In [ ]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# Download required NLTK resources for tokenization, stopwords, and lemmatization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

print("Libraries imported and NLTK resources configured successfully.")

ModuleNotFoundError: No module named 'nltk'

# Loading Text Dataset and Performing Text Preprocessing

In [ ]:
# Sample text corpus for text processing and feature extraction
documents = [
    "Data science involves studying data, machine learning models, and algorithms.",
    "Machine learning algorithms learn patterns from training data to make predictions.",
    "Natural language processing helps machines understand human languages and text.",
    "Data preprocessing and feature scaling improve machine learning model performance."
]

# Set of English stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase the text
    text = text.lower()
    # 2. Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    # 3. Tokenize words
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return " ".join(filtered_tokens)

cleaned_documents = [preprocess_text(doc) for doc in documents]

# Display original vs preprocessed text
for i, (orig, clean) in enumerate(zip(documents, cleaned_documents), 1):
    print(f"Doc {i} Original: {orig}")
    print(f"Doc {i} Cleaned : {clean}\n")

# Applying TF-IDF Vectorization and Displaying Transformed Output

In [ ]:
# TF-IDF Vectorization on the preprocessed text dataset
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_documents)

# Create DataFrame for tabular representation of TF-IDF feature matrix
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(cleaned_documents))]
)

print(f"TF-IDF Vocabulary Size: {len(tfidf_vectorizer.get_feature_names_out())}")
print("TF-IDF Feature Matrix:")
tfidf_df

## 1. Apply CountVectorizer to the same text dataset

In [ ]:
# Applying CountVectorizer (Bag of Words) to the same preprocessed text dataset
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(cleaned_documents)

# Create DataFrame for tabular representation of CountVectorizer matrix
count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(cleaned_documents))]
)

print(f"CountVectorizer Vocabulary Size: {len(count_vectorizer.get_feature_names_out())}")
print("CountVectorizer (Bag of Words) Matrix:")
count_df

## 2. Compare TF-IDF and CountVectorizer outputs

In [ ]:
# 1. Term-by-Term Comparison for Document 1
doc1_comparison = pd.DataFrame({
    'Word': count_vectorizer.get_feature_names_out(),
    'CountVectorizer (Term Frequency)': count_matrix.toarray()[0],
    'TF-IDF Score': np.round(tfidf_matrix.toarray()[0], 4)
})

# Display words present in Doc 1
print("--- Comparison of CountVectorizer vs TF-IDF for Doc 1 Terms ---")
display(doc1_comparison[doc1_comparison['CountVectorizer (Term Frequency)'] > 0].reset_index(drop=True))

# 2. Summary of Key Differences
comparison_summary = pd.DataFrame({
    'Feature': [
        'Output Values',
        'Word Importance Weighting',
        'Impact of Frequent Words',
        'Length Normalization',
        'Primary Application'
    ],
    'CountVectorizer': [
        'Integer counts (raw term frequency: 1, 2, 3...)',
        'Equal weight to all occurrences based solely on count',
        'Common corpus words dominate due to high frequencies',
        'Not normalized (longer documents have higher counts)',
        'Basic text modeling, frequency counts, simple classification'
    ],
    'TF-IDF Vectorizer': [
        'Floating-point scores between 0.0 and 1.0',
        'Penalizes words frequent across multiple documents (IDF)',
        'Domain-specific rare words receive higher significance weights',
        'Applies Euclidean (L2) normalization by default',
        'Information retrieval, document search, text classification'
    ]
})

print("\n--- Summary Comparison Between CountVectorizer and TF-IDF ---")
display(comparison_summary)

## 3. Perform lemmatization instead of stemming

In [ ]:
# Initializing Porter Stemmer and WordNet Lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Representative words extracted from the dataset
sample_words = [
    "studying", "studies", "learning", "algorithms",
    "patterns", "predictions", "languages", "scaling", "improves"
]

records = []
for word in sample_words:
    stemmed = stemmer.stem(word)
    lemmatized_noun = lemmatizer.lemmatize(word, pos='n')
    lemmatized_verb = lemmatizer.lemmatize(word, pos='v')
    records.append({
        'Original Word': word,
        'Porter Stemmer': stemmed,
        'Lemmatizer (Noun)': lemmatized_noun,
        'Lemmatizer (Verb)': lemmatized_verb
    })

comparison_df = pd.DataFrame(records)
print("--- Comparison of Stemming vs Lemmatization on Sample Words ---")
display(comparison_df)

# Applying Lemmatization to the preprocessed dataset
def lemmatize_text(text):
    tokens = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(w) for w in tokens])

lemmatized_documents = [lemmatize_text(doc) for doc in cleaned_documents]

print("\n--- Preprocessed vs Lemmatized Documents ---")
for i, (clean, lem) in enumerate(zip(cleaned_documents, lemmatized_documents), 1):
    print(f"Doc {i} Preprocessed: {clean}")
    print(f"Doc {i} Lemmatized  : {lem}\n")

## 4. Apply Min-Max Scaling to another numerical dataset

In [ ]:
# Creating another numerical dataset with features of varying scales
numerical_data = pd.DataFrame({
    'Age': [22, 25, 47, 52, 36, 29, 61],
    'Salary': [25000, 32000, 85000, 110000, 62000, 48000, 135000],
    'Experience_Years': [1, 3, 20, 25, 12, 5, 32],
    'Credit_Score': [600, 650, 780, 810, 710, 680, 840]
})

print("--- Original Numerical Dataset ---")
display(numerical_data)

# Applying Min-Max Scaling: X_scaled = (X - X_min) / (X_max - X_min)
min_max_scaler = MinMaxScaler()
scaled_data = min_max_scaler.fit_transform(numerical_data)

min_max_df = pd.DataFrame(
    np.round(scaled_data, 4),
    columns=numerical_data.columns,
    index=numerical_data.index
)

print("\n--- Min-Max Scaled Dataset (Bounded within [0, 1]) ---")
display(min_max_df)

print("\n--- Summary Statistics of Min-Max Scaled Data ---")
display(min_max_df.describe().loc[['min', 'max', 'mean', 'std']])

## 5. Compare Normalization and Standardization results

In [ ]:
# 1. Standardization (StandardScaler: mean=0, std=1)
standard_scaler = StandardScaler()
standardized_values = standard_scaler.fit_transform(numerical_data)
standardized_df = pd.DataFrame(
    np.round(standardized_values, 4),
    columns=numerical_data.columns,
    index=numerical_data.index
)

# 2. Normalization (Normalizer: L2 norm per sample row = 1)
normalizer = Normalizer(norm='l2')
normalized_values = normalizer.fit_transform(numerical_data)
normalized_df = pd.DataFrame(
    np.round(normalized_values, 6),
    columns=numerical_data.columns,
    index=numerical_data.index
)

print("--- 1. Original Dataset Statistics ---")
display(numerical_data.describe().loc[['mean', 'std', 'min', 'max']])

print("\n--- 2. Standardization (Z-Score: mean ≈ 0, std = 1) ---")
display(standardized_df)
print("Standardized Statistics:")
display(standardized_df.describe().loc[['mean', 'std', 'min', 'max']])

print("\n--- 3. Normalization (L2 Norm: sample vector length = 1) ---")
display(normalized_df)
print("L2 Norm per row (verification that each sample has unit length = 1.0):")
print(np.round(np.linalg.norm(normalized_values, axis=1), 4))

# 4. Comparison Summary
comparison_table = pd.DataFrame({
    'Aspect': [
        'Mathematical Formula',
        'Resulting Range',
        'Mean & Standard Deviation',
        'Axis of Operation',
        'Outlier Sensitivity',
        'Best Suited Machine Learning Algorithms'
    ],
    'Standardization (StandardScaler)': [
        '(X - μ) / σ',
        'Unbounded (commonly between -3 and +3)',
        'Mean = 0, Standard Deviation = 1',
        'Column-wise (operates feature by feature)',
        'Robust to outliers (preserves relative variance without clipping)',
        'Linear/Logistic Regression, SVM, PCA, Linear Discriminant Analysis'
    ],
    'Normalization (Normalizer)': [
        'X / ||X||_2  (or Min-Max: (X - min) / (max - min))',
        'L2: Unit hypersphere (norm = 1) | Min-Max: [0, 1]',
        'Varies depending on input magnitudes',
        'Row-wise (operates sample by sample in L2) / Column-wise (Min-Max)',
        'Sensitive to outliers (Min-Max compresses range; L2 alters vector angles)',
        'k-Nearest Neighbors (k-NN), K-Means clustering, Cosine Similarity, ANN'
    ]
})

print("\n--- 4. Comparison Summary: Normalization vs Standardization ---")
display(comparison_table)